# 1. Voting Ensemble


### 1a. Voting Regression

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [8]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import VotingRegressor


In [9]:
# Set random seed for reproducibility
np.random.seed(42)
# Generate synthetic regression dataset
X, y = make_regression(n_samples=300, n_features=10, noise=20, random_state=42)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train_scaled.shape}")
print(f"Test set size: {X_test_scaled.shape}")

Training set size: (240, 10)
Test set size: (60, 10)


In [11]:
regression_param_grids = {
    'Ridge': {
        'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
    },
    'KNN': {
        'n_neighbors': [3, 5, 7, 9, 11],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    },
    'Decision Tree': {
        'max_depth': [5, 8, 10, 12, 15, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'SVR': {
        'C': [0.1, 1, 10, 100, 1000],
        'kernel': ['linear', 'rbf', 'poly'],
        'gamma': ['scale', 'auto']
    }
}

In [12]:
for model_name, param_grid in regression_param_grids.items():
    print(model_name)


Ridge
KNN
Decision Tree
SVR


In [13]:
from sklearn.model_selection import GridSearchCV


tuned_models = {'Linear Regression': LinearRegression()} # as LR has no hyperparameters to tune

# for name in ['Ridge', 'KNN', 'Decision Tree', 'SVR']:
for model_name, param_grid in regression_param_grids.items():

    if model_name == 'Ridge':
        model = Ridge()
    elif model_name == 'KNN':
        model = KNeighborsRegressor()
    elif model_name == 'Decision Tree':
        model = DecisionTreeRegressor(random_state=42)
    elif model_name == 'SVR':
        model = SVR()
    
    grid_search = GridSearchCV(
                                model, 
                                regression_param_grids[model_name],
                                cv=5,  # 5-fold cross-validation
                                scoring='r2',
                                n_jobs=-1,
                                verbose=0
                             )

    grid_search.fit(X_train_scaled, y_train)
    
    tuned_models[model_name] = grid_search.best_estimator_

print(tuned_models)

{'Linear Regression': LinearRegression(), 'Ridge': Ridge(alpha=0.1), 'KNN': KNeighborsRegressor(metric='euclidean', weights='distance'), 'Decision Tree': DecisionTreeRegressor(max_depth=10, min_samples_leaf=2, min_samples_split=5,
                      random_state=42), 'SVR': SVR(C=100, kernel='linear')}


In [14]:
voting_reg = VotingRegressor(
    estimators=[
        ('lr', tuned_models['Linear Regression']),
        ('ridge', tuned_models['Ridge']),
        ('knn', tuned_models['KNN']),
        ('dt', tuned_models['Decision Tree']),
        ('svr', tuned_models['SVR'])
    ]
)

voting_reg.fit(X_train_scaled, y_train)
y_pred_voting = voting_reg.predict(X_test_scaled)

# Evaluate Voting Regressor
mse_voting = mean_squared_error(y_test, y_pred_voting)
rmse_voting = np.sqrt(mse_voting)
mae_voting = mean_absolute_error(y_test, y_pred_voting)
r2_voting = r2_score(y_test, y_pred_voting)

In [15]:
print("Voting Regressor with Tuned Models")
print("="*60)
print(f"\nVoting Regressor (Tuned - Average of 5 models):")
print(f"  RMSE: {rmse_voting:.4f}")
print(f"  MAE:  {mae_voting:.4f}")
print(f"  R²:   {r2_voting:.4f}")

Voting Regressor with Tuned Models

Voting Regressor (Tuned - Average of 5 models):
  RMSE: 47.5086
  MAE:  36.4466
  R²:   0.9291


### 1a. Voting Classification


In [19]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns

# Generate synthetic classification dataset
X_class, y_class = make_classification(
    n_samples=400, n_features=15, n_informative=10, 
    n_redundant=3, n_classes=2, random_state=42
)

# Split the data
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42
)

# Scale the features
scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

In [20]:
classification_param_grids = {
    'Logistic Regression': {
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'solver': ['lbfgs', 'liblinear'],
        'penalty': ['l2']
    },
    'SVM': {
        'C': [0.1, 1, 10, 100],
        'kernel': ['linear', 'rbf', 'poly'],
        'gamma': ['scale', 'auto'],
        'degree': [2, 3]
    },
    'Random Forest': {
        'n_estimators': [50, 100, 150],
        'max_depth': [10, 15, 20, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    },
    'Gradient Boosting': {
        'n_estimators': [30, 50, 100],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7],
        'min_samples_split': [2, 5]
    }
}

In [27]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

tuned_models_clf = {} 

# for name in ['Ridge', 'KNN', 'Decision Tree', 'SVR']:
for model_name, param_grid in classification_param_grids.items():

    if model_name == 'Logistic Regression':
        model = LogisticRegression()
    elif model_name == 'SVM':
        model = SVC()
    elif model_name == 'Random Forest':
        model = RandomForestClassifier(random_state=42)
    elif model_name == 'Gradient Boosting':
        model = GradientBoostingClassifier(random_state=42)
    
    grid_search_clf = GridSearchCV(
                                model, 
                                classification_param_grids[model_name],
                                cv=5,  # 5-fold cross-validation
                                scoring='f1',
                                n_jobs=-1,
                             )

    grid_search_clf.fit(X_train_c_scaled, y_train_c)
    
    tuned_models_clf[model_name] = grid_search_clf.best_estimator_

print(tuned_models_clf)

{'Logistic Regression': LogisticRegression(C=0.1, penalty='l2', solver='liblinear'), 'SVM': SVC(C=1, degree=2), 'Random Forest': RandomForestClassifier(max_depth=10, min_samples_leaf=2, n_estimators=150,
                       random_state=42), 'Gradient Boosting': GradientBoostingClassifier(random_state=42)}


In [28]:
voting_clf = VotingClassifier(
    estimators=[
        ('lr', tuned_models_clf['Logistic Regression']),
        ('svm', tuned_models_clf['SVM']),
        ('rf', tuned_models_clf['Random Forest']),
        ('gb', tuned_models_clf['Gradient Boosting'])
    ]
)

voting_clf.fit(X_train_c_scaled, y_train_c)
y_pred_voting = voting_clf.predict(X_test_c_scaled)

# Evaluate Voting Classifier
accuracy_voting = accuracy_score(y_test_c, y_pred_voting)
precision_voting = precision_score(y_test_c, y_pred_voting)
recall_voting = recall_score(y_test_c, y_pred_voting)
f1_voting = f1_score(y_test_c, y_pred_voting)

print("\nVoting Classifier with Tuned Models")
print("="*60)

print(f"\nVoting Classifier (Tuned - Majority Vote of 4 models):")
print(f"  Accuracy:  {accuracy_voting:.4f}")
print(f"  Precision: {precision_voting:.4f}")
print(f"  Recall:    {recall_voting:.4f}")
print(f"  F1 Score:  {f1_voting:.4f}")


Voting Classifier with Tuned Models

Voting Classifier (Tuned - Majority Vote of 4 models):
  Accuracy:  0.8250
  Precision: 0.8636
  Recall:    0.8261
  F1 Score:  0.8444


--------------------------------------------------------------------------------------

# 2.  Ensemble
